# Liora Engine · primera generación real (GPU gratuita, 0 €)

Ejecutar en **Kaggle Notebooks** (gratis, sin tarjeta; requiere verificar el teléfono para activar GPU e Internet):
*Settings → Accelerator: GPU T4 x2 o P100 · Internet: On*. La RAM y la VRAM varían según la sesión: comprobarlas antes de descargar pesos. No hay capacidad garantizada.
Colab gratuito (12,7 GB de RAM) probablemente se queda corto en RAM: no verificado.

Este cuaderno **no simula nada**: si falta GPU, pesos o memoria, el motor se detiene y lo deja escrito en `manifest.json`.
La foto oficial de Liora se sube como *Dataset privado* de Kaggle (nunca al repositorio público).

In [ ]:
# Fallar ANTES de instalar o descargar pesos si Kaggle no ha concedido GPU.
import os, shutil, subprocess
assert shutil.which('nvidia-smi'), 'No hay NVIDIA GPU asignada. Activa GPU en Kaggle y reinicia la sesión.'
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
print(gpu.stdout or gpu.stderr)
assert gpu.returncode == 0 and gpu.stdout.strip(), 'No hay GPU NVIDIA utilizable en esta sesión.'
import torch
assert torch.cuda.is_available(), 'PyTorch no detecta CUDA. No ejecutar descarga de modelos.'
print('CUDA disponible. GPUs:',torch.cuda.device_count())
!free -g && df -h /kaggle/working 2>/dev/null || df -h .


In [ ]:
# Código del motor (solo la carpeta liora/engine del repositorio)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/alvarogavilan/edge-funding.git
%cd edge-funding
!git sparse-checkout set liora/engine
%cd liora/engine
# torch ya viene instalado en Kaggle/Colab: solo se añaden las dependencias del motor
!pip install -q "diffusers>=0.35" "transformers>=4.49" "accelerate>=1.1" ftfy sentencepiece imageio-ffmpeg dlib-bin face_recognition_models==0.3.0 mediapipe
!apt-get -qq install -y libegl1 libgles2 > /dev/null 2>&1 || true

In [ ]:
# 1) Medición real de la capacidad gratuita (se guarda en capacity.json)
!python -m liora_engine probe --bench --out /kaggle/working/capacity.json | head -60

In [ ]:
# 2) Foto oficial (Dataset privado de Kaggle). Ajusta la ruta.
import glob
cands = glob.glob('/kaggle/input/*/*.jp*g') + glob.glob('/kaggle/input/*/*.png') + glob.glob('/kaggle/input/*/*/*.jp*g') + glob.glob('/kaggle/input/*/*/*.png')
print(cands)
OFFICIAL = cands[0] if cands else None
assert OFFICIAL, 'Añade la foto oficial como Dataset privado (Add data → Upload)'

In [ ]:
# 3) Paquete de identidad: solo descriptores + SHA-256, sin imágenes
!python -m liora_engine identity-enroll --pack /kaggle/working/identity-pack.json --image "$OFFICIAL" --official

In [ ]:
# 4) Descarga de pesos de Wan 2.2 TI2V-5B (Apache-2.0) con medición de tiempo y tamaño
import time, os, subprocess
from huggingface_hub import snapshot_download
t0 = time.time()
path = snapshot_download('Wan-AI/Wan2.2-TI2V-5B-Diffusers')
size = int(subprocess.check_output(['du', '-sb', path]).split()[0])
print(path, round(size / 1e9, 1), 'GB en', round(time.time() - t0), 's')
os.environ['LIORA_WAN22_TI2V'] = path

In [ ]:
# 5) Trabajo de 1 escena de 5 s (121 fotogramas, 704×1280, 24 fps) partiendo de la foto oficial
!python -m liora_engine plan --official "$OFFICIAL" --pack /kaggle/working/identity-pack.json --out /kaggle/working/job.json --scenes 1 --seconds 5 --steps 40
!python -m liora_engine models

In [ ]:
# 6) Generación real de la etapa 1 con control de identidad, continuidad y fotogramas
!LIORA_WAN22_TI2V="$path" python -m liora_engine render --job /kaggle/working/job.json --work /kaggle/working/render --only 1
!cat /kaggle/working/render/manifest.json | head -120

In [ ]:
# 7) Verificar que existe un MP4 REAL y que ffprobe lo reconoce. Nada de éxito falso.
import glob, json, subprocess, pathlib, hashlib
from IPython.display import Video, display
clips = sorted(glob.glob('/kaggle/working/render/scene-*/attempt-*.mp4'))
assert clips, 'No existe MP4: revisa render/manifest.json. La generación NO ha concluido.'
clip = pathlib.Path(clips[-1])
info = json.loads(subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=codec_name,width,height,duration','-of','json',str(clip)]))
assert clip.stat().st_size > 10000 and info.get('streams'), 'MP4 inexistente, vacío o ilegible'
print(json.dumps({'file':str(clip),'bytes':clip.stat().st_size,'sha256':hashlib.sha256(clip.read_bytes()).hexdigest(),'streams':info['streams']},indent=2))
display(Video(str(clip), embed=True, width=360))


Si esta prueba termina sin errores, se ha generado un MP4 técnico, **no se ha validado todavía su realismo ni la continuidad del rostro**. Revisa también el manifiesto y las licencias antes de utilizarlo comercialmente. El siguiente paso es encadenar escenas y montar la banda sonora en Liora Studio. La sesión gratuita de Kaggle puede no tener GPU o memoria suficientes.